#

In [12]:
# imports
import pandas as pd
import os

# Helpers
from Scripts.q1k_preprocessing import create_diagnosis_columns, create_IQ_column
from Scripts.q1k_stats import get_group_stats

In [13]:
root_dir = '/Users/emmanuelle.coutu-nadeau/Code/NED LAB/GENiAL'
database = pd.read_csv(os.path.join(root_dir, 'Data/Q1KDatabase-ECNEEGIQGENCHUSJ_DATA_flattened_cleaned_cnv.csv'))

# Table of Contents
1. [Create new diagnosis columns](#Create-new-diagnosis-columns)
2. [Create group column](#create-group-columns)
3. [Sample Overview](#sample-overview)

# Create new diagnosis columns

In [ ]:
output_path = os.path.join(root_dir, 'Data/Q1KDatabase-ECNEEGIQGENCHUSJ_DATA_flattened_cleaned_cnv_renamedcols.csv')

# Apply the function to create new diagnosis columns
create_diagnosis_columns(database, output_path)

database_clean_cols = pd.read_csv(output_path)

# Display the counts for each new diagnosis column
diagnosis_columns = [
    'ASD', 'ASD_behavior', 'ADHD', 'ID', 'OCD', 'motor_disorder',
    'anxiety', 'neurological_conditions', 'genetic_disorder', 'other'
]

print("\nDiagnosis Distribution:")
for col in diagnosis_columns:
    count = database_clean_cols[col].sum()
    percentage = (count / len(database) * 100).round(1)
    print(f"{col}: {count} ({percentage}%)")



In [ ]:
output_path = os.path.join(root_dir, 'Data/Q1KDatabase-ECNEEGIQGENCHUSJ_DATA_flattened_cleaned_cnv_renamedcols_IQ.csv')

# Apply the function to create new diagnosis columns
create_IQ_column(database_clean_cols, output_path)

database_clean_cols_IQ = pd.read_csv(output_path)

# Create Group columns

In [ ]:
database_clean_cols_IQ_groups = database_clean_cols_IQ.copy()

# Create a new Group column with three possible values: Control, Neurodev, Neurodev_gmc
database_clean_cols_IQ_groups['Group'] = 'Control'  # Default value

# Neurodev group (participants with Control=0 and Neurodev=1 and Genetic_carrier = 0)
neurodev_mask = (database_clean_cols_IQ_groups['Control'] == 0) & (database_clean_cols_IQ_groups['Neurodev'] == 1) & (database_clean_cols_IQ_groups['Genetic_carrier'] == 0)
database_clean_cols_IQ_groups.loc[neurodev_mask, 'Group'] = 'Neurodev'

# Neurodev_gmc group (participants with Control=0 and Neurodev=1 and Genetic_carrier=1)
neurodev_gmc_mask = (database_clean_cols_IQ_groups['Control'] == 0) & (database_clean_cols_IQ_groups['Neurodev'] == 1) & (database_clean_cols_IQ_groups['Genetic_carrier'] == 1)
database_clean_cols_IQ_groups.loc[neurodev_gmc_mask, 'Group'] = 'Neurodev_gmc'

# Save the updated dataframe
output_path = os.path.join(root_dir, 'Data/Q1KDatabase-ECNEEGIQGENCHUSJ_DATA_flattened_cleaned_cnv_renamedcols_IQ_groups.csv')
database_clean_cols_IQ_groups.to_csv(output_path, index=False)

# Display group sizes
print("\n-------------- \nGroup Sizes:")
print(f"Control: {len(database_clean_cols_IQ_groups[database_clean_cols_IQ_groups['Group'] == 'Control'])}")
print(f"Neurodev: {len(database_clean_cols_IQ_groups[database_clean_cols_IQ_groups['Group'] == 'Neurodev'])}")
print(f"Neurodev_gmc: {len(database_clean_cols_IQ_groups[database_clean_cols_IQ_groups['Group'] == 'Neurodev_gmc'])}")
print(f"Total: {len(database_clean_cols_IQ_groups)}")

print("\n-------------- \nFamily members count:")
proband_count = database_clean_cols_IQ_groups[database_clean_cols_IQ_groups['participant_id'].str.contains('_P')].shape[0]
print(f"Number of probands: {proband_count}")
sibling_count = database_clean_cols_IQ_groups[database_clean_cols_IQ_groups['participant_id'].str.contains('_S')].shape[0]
print(f"Number of siblings: {sibling_count}")
father_count = database_clean_cols_IQ_groups[database_clean_cols_IQ_groups['participant_id'].str.contains('_F')].shape[0]
print(f"Number of fathers: {father_count}")
mother_count = database_clean_cols_IQ_groups[database_clean_cols_IQ_groups['participant_id'].str.contains('_M')].shape[0]
print(f"Number of mothers: {mother_count}")
other_count = database_clean_cols_IQ_groups[~database_clean_cols_IQ_groups['participant_id'].str.contains('_[PSFM]')].shape[0]
print(f"Number of others: {other_count}")
print(f"Total: {proband_count+sibling_count+father_count+mother_count+other_count}")



# Add demographic info

In [16]:
from Scripts.clean_demographics import clean_demographic_data, merge_demographics_to_main

In [ ]:
# Step 1: Clean the demographic data
root_dir = '/Users/emmanuelle.coutu-nadeau/Code/NED LAB/GENiAL'
input_path = os.path.join(root_dir, 'Data/Q1KDatabase-ECNDEMOG_DATA.csv')
output_path = os.path.join(root_dir, 'Data/Q1KDatabase-ECNDEMOG_DATA_cleaned.csv')

clean_demographic_data(input_path, output_path)

In [ ]:
# Step 2: Merge into your main file
root_dir = '/Users/emmanuelle.coutu-nadeau/Code/NED LAB/GENiAL'
main_path = os.path.join(root_dir, 'Data/Q1KDatabase-ECNEEGIQGENCHUSJ_DATA_flattened_cleaned_cnv_renamedcols_IQ_groups.csv')
demog_path = os.path.join(root_dir, 'Data/Q1KDatabase-ECNDEMOG_DATA_cleaned.csv')
output_path = os.path.join(root_dir, 'Data/Q1KDatabase-ECNEEGIQGENCHUSJ_DATA_flattened_cleaned_cnv_renamedcols_IQ_groups_demog.csv')
merge_demographics_to_main(
    main_path,
    demog_path,
    output_path
)

# Sample Overview

In [ ]:
# Create a table to store results
results = []

# For each diagnosis column
for diagnosis in diagnosis_columns:
    row = {'Diagnosis': diagnosis}
    
    # Get stats for Control group
    count, pct, total = get_group_stats(database_clean_cols_IQ, 'Control', diagnosis)
    row['Control'] = f"{count} ({pct}%)"
    
    # Get stats for Neurodev without Genetic_carrier
    neurodev_no_genetic = database_clean_cols_IQ[(database_clean_cols_IQ['Neurodev'] == 1) & (database_clean_cols_IQ['Genetic_carrier'] == 0)]
    total = len(neurodev_no_genetic)
    count = neurodev_no_genetic[diagnosis].sum()
    pct = (count / total * 100).round(1) if total > 0 else 0
    row['Neurodev without Genetic_carrier'] = f"{count} ({pct}%)"
    
    # Get stats for Neurodev AND Genetic_carrier
    neurodev_genetic = database_clean_cols_IQ[(database_clean_cols_IQ['Neurodev'] == 1) & (database_clean_cols_IQ['Genetic_carrier'] == 1)]
    total = len(neurodev_genetic)
    count = neurodev_genetic[diagnosis].sum()
    pct = (count / total * 100).round(1) if total > 0 else 0
    row['Neurodev AND Genetic_carrier'] = f"{count} ({pct}%)"
    
    results.append(row)

# Convert to DataFrame and display
results_df = pd.DataFrame(results)

# Add sample sizes to column headers
control_total = database_clean_cols_IQ['Control'].sum()
neurodev_no_genetic_total = len(database_clean_cols_IQ[(database_clean_cols_IQ['Neurodev'] == 1) & (database_clean_cols_IQ['Genetic_carrier'] == 0)])
neurodev_genetic_total = len(database_clean_cols_IQ[(database_clean_cols_IQ['Neurodev'] == 1) & (database_clean_cols_IQ['Genetic_carrier'] == 1)])

results_df.columns = ['Diagnosis', 
                     f'Control [n={control_total}]', 
                     f'Neurodev without Genetic_carrier [n={neurodev_no_genetic_total}]', 
                     f'Neurodev AND Genetic_carrier [n={neurodev_genetic_total}]']

print("\nDiagnosis Distribution by Group:")
display(results_df)

# Add age, sex, and IQ information for each group
print("\nAge, Sex, and IQ Distribution by Group:")
demographic_results = []

for group in ['Control', 'Neurodev without Genetic_carrier', 'Neurodev AND Genetic_carrier']:
    if group == 'Neurodev AND Genetic_carrier':
        group_data = database_clean_cols_IQ[(database_clean_cols_IQ['Neurodev'] == 1) & (database_clean_cols_IQ['Genetic_carrier'] == 1)]
    elif group == 'Neurodev without Genetic_carrier':
        group_data = database_clean_cols_IQ[(database_clean_cols_IQ['Neurodev'] == 1) & (database_clean_cols_IQ['Genetic_carrier'] == 0)]
    else:
        group_data = database_clean_cols_IQ[database_clean_cols_IQ[group] == 1]
    
    # Calculate age statistics
    age_mean = group_data['eeg_test_age'].mean()
    age_std = group_data['eeg_test_age'].std()
    
    # Calculate sex distribution
    sex_counts = group_data['sex'].value_counts()
    total = len(group_data)
    male_pct = (sex_counts.get('M', 0) / total * 100).round(1)
    female_pct = (sex_counts.get('F', 0) / total * 100).round(1)
    
    # Calculate IQ statistics
    iq_mean = group_data['IQ'].mean()
    iq_std = group_data['IQ'].std()
    
    demographic_results.append({
        'Group': group,
        'Age (mean ± std)': f"{age_mean:.1f} ± {age_std:.1f}",
        'Male': f"{sex_counts.get('M', 0)} ({male_pct}%)",
        'Female': f"{sex_counts.get('F', 0)} ({female_pct}%)",
        'IQ (mean ± std)': f"{iq_mean:.1f} ± {iq_std:.1f}"
    })

# Display demographic results
demographic_df = pd.DataFrame(demographic_results)
display(demographic_df)
